[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Vision and Language - Image & Text Embedding

This notebooks shows how to use Large Vision Model for meta data extraction from a football game frame.

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 13/10/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2026_02/0130DeepLearningDiffusion.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Image Processing
import imageio.v3 as iio

# Deep Learning
from transformers import AutoTokenizer, AutoModel

# Miscellaneous
import json
import os
import random

# Typing
from typing import Callable, Dict, Literal, Optional, Tuple, Union
from numpy.typing import NDArray

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython
from IPython.display import JSON, Markdown, display

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages


In [ ]:
# General Auxiliary Functions

def ParseEnvFile( fileName: str = '.env', *, filePath: str = '.', keyValSep: str = '=' ) -> Dict[str, str]:
    # Read the file and parse it into a dictionary
    dEnv = {}
    with open(os.path.join(filePath, fileName), 'r') as hF:
        for line in hF:
            if line.startswith('#') or not line.strip():
                continue
            key, value = line.strip().split(keyValSep, 1)
            dEnv[key]  = value.strip()
    return dEnv

## Multi Modal Large Models

Many current LLMs are _Multi Modal Models_.  
A _Modality_ is a type of data: _Text_, _Image_, _Audio_ and _Video_.

A multi modal model can process more than one modality.  
For example, it can receive an image together with a text question.  
It can then answer the question using information from the image.

### Combining Modalities

Each modality must be converted into a numerical representation.  
An image encoder extracts visual features.  
Text is represented using token embeddings.  
The model combines these representations to reason about the inputs.

One common approach maps different modalities into a _Shared Embedding Space_.  
Related content can then have similar representations.  
For example, an _image of a football_ and the word _"football"_ can be aligned.  

* <font color='brown'>(**#**)</font> [OpenAI Developers page shows a comparisons of models](https://developers.openai.com/api/docs/models/compare) with their supported modalities.

### LLM API

Most LLM APIs use 2 main message types:

 - System Prompt: Defines the model's role, rules, and response style.
 - User Prompt: Provides the task and its data. It may include text or an image.

Model usage may be billed in 4 categories:

 - Input: New input processed by the model. Includes instructions and task data.
 - Output: Content generated by the model.
 - Cached Input: Previously cached prompt content reused by the model. Usually costs less.
 - Cached Write: Storing prompt content in a cache for later reuse.

A _Session_ groups related requests and their conversation history.  
Each request uses a chosen **model** and, when supported, an **effort level**.  
The model determines the maximum **context window** size.  
The same _system message_ is usually kept throughout the session.

* <font color='brown'>(**#**)</font> The context window is the maximum token budget for one request.  
  In a self attention layer, each token has a _Query_, a _Key_, and a _Value_.  
  During generation, the current token's query is compared with keys from earlier tokens and itself.  
  These matches determine the weights used to combine their values.  
  The window budget includes input and generated tokens.
* <font color='brown'>(**#**)</font> The effort level is a setting for how much reasoning a supported _Thinking Model_ performs before answering.  
  The model can generate intermediate reasoning steps, called _Chain of Thought_.  
  These steps help break a task into smaller parts. They may remain hidden from the user.  
  Higher effort usually allows more reasoning tokens. It can increase response time and cost.  
  More reasoning does not guarantee a correct answer.
* <font color='brown'>(**#**)</font> Text usage is measured in _Tokens_, not words. A token may be part of a word. Images also contribute to input usage.
* <font color='brown'>(**#**)</font> Prices depend on the model and provider. Not all APIs charge separately for cache writes.

## Structured Response

A _Structured Response_ follows a predefined _Schema_.  
The schema defines field names, data types, and allowed values.  
The model returns data, usually as _JSON_, instead of free text.

### Motivation

 - Reduce Hallucinations: Limit the response to the requested fields.  
   Allow `null` for missing or unreadable data instead of forcing a guess.
 - API Integration: Match the fields and types expected by other structured APIs.
 - Reliable Processing: Read values directly. Avoid extracting information from prose.
 - Validation: Check required fields, types, and allowed values before using the response.
 - Consistency: Use the same structure for every frame. Store and compare results easily.
 - Costs: Predictable cost per answer.

### Example

The task is to update a match tracking system from a football frame.  
Its API expects named fields for the time, score, added time, and match period.  
The score follows the _left to right_ order of the scoreboard.

#### Regular Response

The input prompt: "Read the match time, score, added time, and period from this frame."

The model might answer:

> The clock shows 90:23 in the second half. The score is 2-1, left to right. Five minutes of added time are indicated.

A person can read this easily. The API cannot use the sentence directly.  
Our code must extract the values and map them to the API's fields.  
The next response might use different wording. Our parsing logic must handle it.

#### Structured Response

We ask for the same information, using a predefined response structure:

```json
{
  "matchTime": "90:23",
  "score": {"left": 2, "right": 1},
  "addedTimeMinutes": 5,
  "period": "second_half"
}
```

* <font color='brown'>(**#**)</font> The structure can define what type of values are valid (Numbers, text, choice, etc...).
* <font color='brown'>(**#**)</font> Asking for JSON in a prompt is not the same as enforcing a schema.  
Structured response must be defined by a schema with supported API.
* <font color='brown'>(**#**)</font> A valid structure does not guarantee correct facts. Yet it might reduce the hallucinations.

In [ ]:
# Parameters

# Data
dataDirName = 'MetaDataExtractFootballFrames' #<! https://huggingface.co/datasets/Royi/DataSets -> `MetaDataExtractFootballFrames.zip`
lImgFileExt = ['.jpg', '.png']

# Environment
envFile = '.env'

# Model
apiKey            = 'OPENAI_API_KEY'
modelName         = 'gpt-6-luna' #<! GPT 6 Luna
# modelName         = 'gpt-5.4-nano' #<! GPT 6 Luna
contextWindowSize = 65_536 #<! Context window size for the selected model
reasoningEffort   = 'none' #<! Default reasoning effort for the selected model
modelTemp         = 0.0 #<! Value in range [0, 2] where lower values are more deterministic

## The Task: Extracting Metadata from a Football Game Frame

![](https://i.imgur.com/uILF2bF.png)
<!-- ![](https://i.postimg.cc/KjMjZXqg/Frame00029.png) -->

Commonly, the scoreboard in a _football game_ (Soccer) frame shows:
 - Clock: The displayed match time. Usually minutes and seconds, such as `67:42`.
 - Result: The current number of goals scored by each team, such as `2 - 1`.
 - Added Time / Stoppage Time: Time to compensate for stoppages at the end of a period. An indicator such as `+5` means at least 5 added minutes.
 - Extra Time: Knockout matches use it when the score is tied after normal time. Usually 2 halves of 15 minutes.

The task is to read these values from the frame. The required data:

 - The match period (First half, second half, or either half of extra time).
 - The time.
 - The result.
 - The _Stoppage Time_.

* <font color='red'>(**?**)</font> Think of valid values per filed.
* <font color='brown'>(**#**)</font> A _Structured Response_ is well suited for the task above. 
* <font color='brown'>(**#**)</font> The response should handle the cases the model can not extract the required metadata.

## Generate / Load Data

 - Download the file `MetaDataExtractFootballFrames.zip` from https://huggingface.co/datasets/Royi/DataSets.
 - Unpack it into `FixelCourses\DataSets`.
 - Verify images in `FixelCourses\DataSets\MetaDataExtractFootballFrames`.

In [ ]:
# Generate / Load Data

dataDirPath = os.path.join(DATA_FOLDER_PATH, dataDirName)

lImgFile = os.listdir(dataDirPath)
lImgFile = [f for f in lImgFile if os.path.splitext(f)[1].lower() in lImgFileExt]
numSamples = len(lImgFile)

print(f'Number of images: {numSamples}')

In [ ]:
# Environment

# The `.env` should have a row with:
# ```
# OPENAI_API_KEY=<your_openai_api_key_here>
# ```

dEnv = ParseEnvFile(envFile)

### Plot the Data

In [ ]:
# Plot the Data

hF, vHa = plt.subplots(nrows = 3, ncols = 3, figsize = (12, 8))
vHa = vHa.flat

for ii, hA in enumerate(vHa):
    randIdx = random.randint(0, numSamples - 1)
    imgFileName = lImgFile[randIdx]
    imgFilePath = os.path.join(dataDirPath, imgFileName)

    tI = iio.imread(imgFilePath)
    
    hA.imshow(tI)
    hA.set_title(f'Label: {imgFileName}')
    hA.axis('off')

hF.tight_layout()

* <font color='red'>(**?**)</font> Is the whole image needed in order to accomplish the task? Think about optimizing the cost.

## Prose Session


In [ ]:
# Session

oSessionProse = LLMSessionProse(dEnv[apiKey], modelName, contextWindowBudget = contextWindowSize, reasoningEffort = reasoningEffort, valTemp = modelTemp)

In [ ]:
# Load Image

imgIdx = 123
imgFileName = lImgFile[imgIdx]
imgFilePath = os.path.join(dataDirPath, imgFileName)
tI = iio.imread(imgFilePath)

hF, hA = plt.subplots(figsize = (8, 6))
hA.imshow(tI)
hA.set_title(f'Label: {imgFileName}')
hA.axis('off');

In [ ]:
# Pre Process Image

numRows, numCols, numChannels = tI.shape

tX = tI[:(numRows // 2), :(numCols // 2), :]
tX = sp.ndimage.zoom(tX, (0.5, 0.5, 1.0))

hF, hA = plt.subplots(figsize = (8, 6))
hA.imshow(tX)
hA.set_title(f'Label: {imgFileName}')
hA.axis('off');

In [ ]:
# Prompt

promptMsg = "The input image is a frame from a video of a football game. The task is to extract the match period, the time. the score, the stoppage time and the time within the stoppage time."

In [ ]:
# Answer

answerMsg = oSessionProse.Prompt(promptMsg, tX)
display(Markdown(answerMsg))

* <font color='brown'>(**#**)</font> Modern day LLM's are so good they "predict" the intention.

## Structured Session

In [ ]:
# Structured Response Schema

systemMsg = (
    "Extract match metadata from the scoreboard in this football frame. Follow the response schema. "
    "Use only visible information. Do not guess. Use null for missing or unreadable values. "
    "Set `has_time` to false when no readable match clock is visible. "
    "Read scores from left to right or top to bottom. "
    "Distinguish extra time from stoppage time. Record announced added time and elapsed stoppage time separately. "
    "Set `confidence_level` to 'None' if nothing can be extracted, 'Low' for uncertain readings or 'High' for clear readings."
)

dSchema = {
    "type": "object",
    "properties": {
        "match_period": {
            "type": ["string", "null"],
            "enum": ["1st Half", "2nd Half", "1st Extratime", "2nd Extratime"],
            "description": "Match period identified from visible information, or null when unavailable.",
        },
        "has_time": {
            "type": "boolean",
            "description": "Whether a match clock is visible in the frame.",
        },
        "time_mm_ss": {
            "type": ["string", "null"],
            "description": "Visible match time in MM:SS format, or null when unavailable.",
            "pattern": r"^\d{2}:\d{2}$",
        },
        "confidence_level": {
            "type": "string",
            "enum": ["None", "Low", "High"],
            "description": "Overall extraction confidence. None: no requested metadata can be extracted. Low: uncertain extraction. High: extracted values are clear.",
        },
        "stoppage_time_referee": {
            "type": ["string", "null"],
            "description": "Visible stoppage time decided by the referee in +X format, or null when unavailable.",
            "pattern": r"^\+\d+$",
        },
        "stoppage_time_actual": {
            "type": ["string", "null"],
            "description": "Visible actual stoppage time in MM:SS (e.g. `02:19`) or M:SS (e.g. `2:19`) format, or null when unavailable.",
            "pattern": r"^\d{1,2}:\d{2}$",
        },
        "first_team_score": {
            "type": ["integer", "null"],
            "description": "Visible score of the first team, or null when unavailable.",
            "minimum": 0,
            "maximum": 99,
        },
        "second_team_score": {
            "type": ["integer", "null"],
            "description": "Visible score of the second team, or null when unavailable.",
            "minimum": 0,
            "maximum": 99,
        },
    },
    "required": [
        "match_period", "has_time", "time_mm_ss", "confidence_level", "stoppage_time_referee",
        "stoppage_time_actual", "first_team_score", "second_team_score",
    ],
    "additionalProperties": False,
}

promptMsg = "Extract the match metadata from the scoreboard in this football frame following the structured response schema."

In [ ]:
# Structured Response Section

oSessionStructured = LLMSessionStructured(dEnv[apiKey], modelName, dSchema = dSchema, systemMsg = systemMsg, contextWindowBudget = contextWindowSize, reasoningEffort = reasoningEffort, valTemp = modelTemp)

In [ ]:
# Answer

answerMsg = oSessionStructured.Prompt(promptMsg, tX)
display(Markdown(f'```json\n{json.dumps(answerMsg, indent = 4, ensure_ascii = False)}\n```'))

In [ ]:
hF, hA = plt.subplots(figsize = (8, 6))
hA.imshow(tX)
hA.set_title(f'Label: {imgFileName}')
hA.axis('off');

In [ ]:
# Close Sessions

oSessionProse.Close()
oSessionStructured.Close()

* <font color='green'>(**@**)</font> Build a class and a script to process the whole folder. MNind the token usage. Save to output folder with the output result for analysis.